In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the RAW data again — we redo cleaning inside the pipeline to avoid leakage
df = pd.read_csv("../data/raw/uci-secom.csv")
y = df["Pass/Fail"]
X = df.drop(columns=["Time", "Pass/Fail"])

# Drop zero-variance and high-missing columns (these are safe — structural, not learned from values)
missing_pct = X.isnull().mean()
X = X.drop(columns=missing_pct[missing_pct > 0.5].index)
X = X.drop(columns=X.columns[X.nunique() <= 1])

print("Feature matrix before split:", X.shape)

Feature matrix before split: (1567, 446)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # <-- critical
    random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).round(3))

Train: (1253, 446) | Test: (314, 446)

Train class balance:
Pass/Fail
-1    0.934
 1    0.066
Name: proportion, dtype: float64

Test class balance:
Pass/Fail
-1    0.933
 1    0.067
Name: proportion, dtype: float64


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

pipe.fit(X_train, y_train)
print("Pipeline trained.")

Pipeline trained.


In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipe.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

Confusion matrix:
[[271  22]
 [ 18   3]]

Classification report:
              precision    recall  f1-score   support

          -1      0.938     0.925     0.931       293
           1      0.120     0.143     0.130        21

    accuracy                          0.873       314
   macro avg      0.529     0.534     0.531       314
weighted avg      0.883     0.873     0.878       314



In [5]:
pipe_balanced = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

pipe_balanced.fit(X_train, y_train)
y_pred_bal = pipe_balanced.predict(X_test)

print("Confusion matrix (balanced):")
print(confusion_matrix(y_test, y_pred_bal))
print("\nClassification report (balanced):")
print(classification_report(y_test, y_pred_bal, digits=3))

Confusion matrix (balanced):
[[259  34]
 [ 17   4]]

Classification report (balanced):
              precision    recall  f1-score   support

          -1      0.938     0.884     0.910       293
           1      0.105     0.190     0.136        21

    accuracy                          0.838       314
   macro avg      0.522     0.537     0.523       314
weighted avg      0.883     0.838     0.859       314



In [6]:
from sklearn.ensemble import RandomForestClassifier

pipe_rf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
])

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print("Confusion matrix (Random Forest):")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification report (Random Forest):")
print(classification_report(y_test, y_pred_rf, digits=3))

Confusion matrix (Random Forest):
[[293   0]
 [ 21   0]]

Classification report (Random Forest):
              precision    recall  f1-score   support

          -1      0.933     1.000     0.965       293
           1      0.000     0.000     0.000        21

    accuracy                          0.933       314
   macro avg      0.467     0.500     0.483       314
weighted avg      0.871     0.933     0.901       314



/Users/janiceivana/wafer-escape-prevention/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/janiceivana/wafer-escape-prevention/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/janiceivana/wafer-escape-prevention/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [7]:
# Probability of class "1" (failure) for each test wafer
# Find which column corresponds to class 1
classes = pipe_rf.named_steps["model"].classes_
print("Classes:", classes)   # tells us column order, e.g. [-1, 1]

fail_idx = list(classes).index(1)
y_scores = pipe_rf.predict_proba(X_test)[:, fail_idx]

print("Score range:", y_scores.min().round(3), "to", y_scores.max().round(3))

Classes: [-1  1]
Score range: 0.02 to 0.383


In [8]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_test, y_scores, pos_label=1
)

# Show the tradeoff at a range of thresholds
import pandas as pd
tradeoff = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],   # curve returns one extra point
    "recall": recall[:-1],
})

# Look at lower thresholds where failures actually get flagged
print(tradeoff[(tradeoff.threshold >= 0.10) & (tradeoff.threshold <= 0.35)]
      .round(3).to_string(index=False))

 threshold  precision  recall
     0.100      0.098   0.905
     0.103      0.102   0.905
     0.107      0.104   0.905
     0.110      0.107   0.905
     0.113      0.110   0.905
     0.117      0.115   0.905
     0.120      0.112   0.857
     0.123      0.107   0.762
     0.127      0.109   0.762
     0.130      0.114   0.762
     0.133      0.113   0.714
     0.137      0.125   0.714
     0.140      0.133   0.714
     0.143      0.143   0.714
     0.147      0.143   0.667
     0.150      0.140   0.619
     0.153      0.149   0.619
     0.157      0.157   0.619
     0.160      0.162   0.619
     0.163      0.167   0.619
     0.167      0.178   0.619
     0.170      0.181   0.619
     0.173      0.176   0.571
     0.177      0.190   0.571
     0.180      0.197   0.571
     0.183      0.200   0.524
     0.187      0.216   0.524
     0.193      0.239   0.524
     0.200      0.227   0.476
     0.207      0.244   0.476
     0.210      0.263   0.476
     0.213      0.286   0.476
     0.217

In [18]:
chosen_threshold = 0.25   # <-- put YOUR chosen value here
y_pred_tuned = (y_scores >= chosen_threshold).astype(int)
# convert back to -1/1 labels to match y_test
y_pred_tuned = np.where(y_pred_tuned == 1, 1, -1)

from sklearn.metrics import confusion_matrix, classification_report
print("Confusion matrix (tuned threshold):")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification report (tuned threshold):")
print(classification_report(y_test, y_pred_tuned, digits=3))

Confusion matrix (tuned threshold):
[[277  16]
 [ 15   6]]

Classification report (tuned threshold):
              precision    recall  f1-score   support

          -1      0.949     0.945     0.947       293
           1      0.273     0.286     0.279        21

    accuracy                          0.901       314
   macro avg      0.611     0.616     0.613       314
weighted avg      0.903     0.901     0.902       314

